<a href="https://colab.research.google.com/github/VaaniSharma13/Digital_Burnout_Risk/blob/main/Burnout_Risk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("Digital_Burnout_Risk - Form responses 1.csv")

print("Original shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())



Original shape: (53, 20)

Columns:
['Timestamp', 'Age', 'Gender', 'Occupation', 'work_mode', 'Screen_Time', 'Sleep_Hours', 'Exercise_frequency', 'Break_Frequency', 'Work_Study_Hours', 'Commute_Time', 'Mental_Exhaustion', 'Work_Life_Imbalance', 'Emotional_Drain', 'Wakeup_Tired', 'Deadline_Stress', 'Routine_Satisfaction', 'Overall_Stress', 'Social_Time', 'Continue_When_Tired']


In [6]:
# DATA CLEANING
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip()

df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

df = df[(df["Age"] >= 18) & (df["Age"] <= 55)].copy()

print("\nShape after keeping Age 18-55:", df.shape)

df["Exercise_frequency"] = df["Exercise_frequency"].replace({
    "0": "0 days"
})

df["Work_Study_Hours"] = df["Work_Study_Hours"].replace({
    "4–6 hours": "4-6 hours",
    "7–9 hours": "7-9 hours",
    "10–12 hours": "10-12 hours"
})

df["Occupation"] = df["Occupation"].replace({
    "Teacher ": "Teacher",
    "A teacher ": "Teacher",
    "Marine Engineer ": "Marine Engineer",
    "Marine engineer ": "Marine Engineer"
})


Shape after keeping Age 18-55: (50, 20)


In [8]:
# SCORE ASSIGNMENT FOR COLS
screen_map = {
    "less than 3 hrs": 1,
    "3-5 hrs": 2,
    "6-8 hrs": 3,
    "9-11 hrs": 4,
    "more than 11 hrs": 5
}
sleep_map = {
    "more than 8 hrs": 1,
    "7-8 hrs": 2,
    "6-7 hrs": 3,
    "5-6 hrs": 4,
    "less than 5 hrs": 5
}
exercise_map = {
    "5+ days": 1,
    "3-4 days": 2,
    "1-2 days": 3,
    "0 days": 4
}
break_map = {
    "very often": 1,
    "often": 2,
    "sometimes": 3,
    "rarely": 4
}
work_hours_map = {
    "Less than 4 hours": 1,
    "4-6 hours": 2,
    "7-9 hours": 3,
    "10-12 hours": 4,
    "More than 12 hours": 5
}
commute_map = {
    "none": 1,
    "less than 30 mins": 2,
    "30-60 mins": 3,
    "1-2 hrs": 4,
    "more than 2 hours": 5
}
social_map = {
    "5+ times": 1,
    "3-4 times/week": 2,
    "1-2 times/week": 3,
    "rarely": 4
}
continue_map = {
    "rarely": 1,
    "sometimes": 2,
    "yes, almost always": 3
}

# score cols
df["Screen_Score"] = df["Screen_Time"].map(screen_map)
df["Sleep_Score"] = df["Sleep_Hours"].map(sleep_map)
df["Exercise_Score"] = df["Exercise_frequency"].map(exercise_map)
df["Break_Score"] = df["Break_Frequency"].map(break_map)
df["WorkHours_Score"] = df["Work_Study_Hours"].map(work_hours_map)
df["Commute_Score"] = df["Commute_Time"].map(commute_map)
df["Social_Score"] = df["Social_Time"].map(social_map)
df["Continue_Score"] = df["Continue_When_Tired"].map(continue_map)

# rating cols to numeric
rating_cols = [
    "Mental_Exhaustion",
    "Work_Life_Imbalance",
    "Emotional_Drain",
    "Wakeup_Tired",
    "Deadline_Stress",
    "Routine_Satisfaction",
    "Overall_Stress"
]

for col in rating_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# satisfactn score rev
df["Routine_Satisfaction_Rev"] = 6 - df["Routine_Satisfaction"]

In [9]:
# ASSIGNING BURNOUT SCORE AND LABELS
df["BurnoutScore"] = (
    df["Screen_Score"] +
    df["Sleep_Score"] +
    df["Exercise_Score"] +
    df["Break_Score"] +
    df["WorkHours_Score"] +
    df["Commute_Score"] +
    df["Mental_Exhaustion"] +
    df["Work_Life_Imbalance"] +
    df["Emotional_Drain"] +
    df["Wakeup_Tired"] +
    df["Deadline_Stress"] +
    df["Routine_Satisfaction_Rev"] +
    df["Overall_Stress"] +
    df["Social_Score"] +
    df["Continue_Score"]
)
def burnout_label(score):
    if score <= 30:
        return "Low"
    elif score <= 45:
        return "Moderate"
    else:
        return "High"

df["Burnout_Risk"] = df["BurnoutScore"].apply(burnout_label)

print("\nBurnout risk counts:")
print(df["Burnout_Risk"].value_counts())



Burnout risk counts:
Burnout_Risk
Moderate    27
High        18
Low          5
Name: count, dtype: int64


In [10]:
# CLEANED DATAASET
df.to_csv("burnout_cleaned_with_labels.csv", index=False)
print("\nSaved: burnout_cleaned_with_labels.csv")



Saved: burnout_cleaned_with_labels.csv


In [11]:
# FEATURES AND TARGET
drop_cols = [
    "Timestamp",
    "BurnoutScore",
    "Burnout_Risk",
    "Screen_Score",
    "Sleep_Score",
    "Exercise_Score",
    "Break_Score",
    "WorkHours_Score",
    "Commute_Score",
    "Social_Score",
    "Continue_Score",
    "Routine_Satisfaction_Rev"
]

X = df.drop(columns=drop_cols)
y = df["Burnout_Risk"]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("\nCategorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)



Categorical columns: ['Gender', 'Occupation', 'work_mode', 'Screen_Time', 'Sleep_Hours', 'Exercise_frequency', 'Break_Frequency', 'Work_Study_Hours', 'Commute_Time', 'Social_Time', 'Continue_When_Tired']
Numerical columns: ['Age', 'Mental_Exhaustion', 'Work_Life_Imbalance', 'Emotional_Drain', 'Wakeup_Tired', 'Deadline_Stress', 'Routine_Satisfaction', 'Overall_Stress']


In [13]:
# DATA PREPROCESSING
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [15]:
# TRAIN TEST
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [17]:
# MODELS IN USE
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

best_model = None
best_accuracy = 0
best_name = ""

for name, model in models.items():
    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(name)
    print("Accuracy:", round(acc, 4))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = clf
        best_name = name

print("\nBest Model:", best_name)
print("Best Accuracy:", round(best_accuracy, 4))

Logistic Regression
Accuracy: 0.5

Classification Report:
              precision    recall  f1-score   support

        High       0.50      0.25      0.33         4
         Low       0.50      1.00      0.67         1
    Moderate       0.50      0.60      0.55         5

    accuracy                           0.50        10
   macro avg       0.50      0.62      0.52        10
weighted avg       0.50      0.50      0.47        10

Confusion Matrix:
[[1 0 3]
 [0 1 0]
 [1 1 3]]
Decision Tree
Accuracy: 0.4

Classification Report:
              precision    recall  f1-score   support

        High       0.33      0.25      0.29         4
         Low       0.50      1.00      0.67         1
    Moderate       0.40      0.40      0.40         5

    accuracy                           0.40        10
   macro avg       0.41      0.55      0.45        10
weighted avg       0.38      0.40      0.38        10

Confusion Matrix:
[[1 0 3]
 [0 1 0]
 [2 1 2]]
Random Forest
Accuracy: 0.6

Classif

In [18]:
# FACTORS AFFECTING BURNOUT
if best_name == "Random Forest":
    ohe = best_model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
    cat_feature_names = ohe.get_feature_names_out(categorical_cols)
    all_feature_names = numerical_cols + list(cat_feature_names)

    importances = best_model.named_steps["model"].feature_importances_

    feat_imp = pd.DataFrame({
        "Feature": all_feature_names,
        "Importance": importances
    }).sort_values(by="Importance", ascending=False)

    print("\nTop 10 Important Features:")
    print(feat_imp.head(10))

    feat_imp.to_csv("feature_importance.csv", index=False)
    print("\nSaved: feature_importance.csv")


Top 10 Important Features:
                                   Feature  Importance
2                      Work_Life_Imbalance    0.122476
7                           Overall_Stress    0.078145
56           Continue_When_Tired_sometimes    0.070315
57  Continue_When_Tired_yes, almost always    0.069902
4                             Wakeup_Tired    0.064075
1                        Mental_Exhaustion    0.046427
0                                      Age    0.037119
5                          Deadline_Stress    0.032369
3                          Emotional_Drain    0.031593
6                     Routine_Satisfaction    0.028448

Saved: feature_importance.csv
